<a href="https://colab.research.google.com/github/AlexitoFernandez/practicas-google-colab/blob/unidad-3/Pr%C3%A1ctica_2_Regresi%C3%B3n_Log%C3%ADstica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#"Machine Learning"
##Unidad III
### **Practica 2 - Regresión Logística**

Alumno: Jorge Alejandro Fernández De Los Santos.

Facilitador: José Gabriel Rodríguez Rivas.

In [32]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import gdown

# Cargar el dataset
file_id = '1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob'
output = 'lending_club_2007_2011_6_states.csv'

# Descargar usando el ID
gdown.download(id=file_id, output=output, quiet=False)
prestamos_df= pd.read_csv(output)
print(prestamos_df.head())

Downloading...
From: https://drive.google.com/uc?id=1HBU_efbbPg378pZvLZ1AtvRFAKrEFsob
To: /content/lending_club_2007_2011_6_states.csv
100%|██████████| 7.01M/7.01M [00:00<00:00, 257MB/s]


   loan_amnt  funded_amnt  funded_amnt_inv       term  int_rate  installment  \
0       2400         2400           2400.0  36 months     15.96        84.33   
1      10000        10000          10000.0  36 months     13.49       339.31   
2       3000         3000           3000.0  36 months     18.64       109.43   
3       5600         5600           5600.0  60 months     21.28       152.39   
4       5375         5375           5350.0  60 months     12.69       121.45   

  grade sub_grade            emp_title emp_length  ... application_type  \
0     C        C5                  NaN  10+ years  ...       Individual   
1     C        C1  AIR RESOURCES BOARD  10+ years  ...       Individual   
2     E        E1      MKC Accounting     9 years  ...       Individual   
3     F        F2                  NaN    4 years  ...       Individual   
4     B        B5            Starbucks   < 1 year  ...       Individual   

   acc_now_delinq chargeoff_within_12_mths delinq_amnt pub_rec_bankr

In [36]:
# Encode categorical features into numerical _code columns
# These columns are often needed for machine learning models.

# Ensure these columns exist before attempting to encode them
if 'grade' in prestamos_df.columns: prestamos_df['grade_code'] = prestamos_df['grade'].factorize()[0]
if 'purpose' in prestamos_df.columns: prestamos_df['purpose_code'] = prestamos_df['purpose'].factorize()[0]
if 'addr_state' in prestamos_df.columns: prestamos_df['addr_state_code'] = prestamos_df['addr_state'].factorize()[0]
if 'home_ownership' in prestamos_df.columns: prestamos_df['home_ownership_code'] = prestamos_df['home_ownership'].factorize()[0]

print("Categorical columns encoded successfully.")

Categorical columns encoded successfully.


In [37]:
#Los meses al estar en stack, se transforman de texto a número y después se dividen en 12 para transformarlos a años
prestamos_df['loan_term_year'] = prestamos_df['term'].str.extract(r'(\d+)').astype(int) / 12

#La Regresión Logística es un clasificador binario, por lo que necesita convertir estas categorías en algo que pueda medir
prestamos_df['repaid'] = prestamos_df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# Selección de variables predictoras
X = prestamos_df[['funded_amnt', 'loan_term_year', 'int_rate', 'grade_code',
                  'purpose_code', 'addr_state_code', 'home_ownership_code',
                  'annual_inc', 'dti', 'revol_util', 'pub_rec_bankruptcies']]

# Variable objetivo o variable a predecir
y = prestamos_df["repaid"]

In [38]:
#División del dataset (60% entrenamiento, 40% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

#Verificación de formas
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11944, 11), (7964, 11), (11944,), (7964,))

In [47]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#Creación y entrenamiento de modelo de Regresión logística sin balanceo de clases
clf1 = LogisticRegression(random_state=0)

#Si falta un dato en alguna columna, el imputer lo reemplaza por el promedio de esa columna.
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)

#Transforma todas las variables para que tengan una media de 0 y una desviación estándar de 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)

#el modelo toma los datos limpios e imputados y los compara con los resultados reales
#para ajustar sus coeficientes matemáticos y poder predecir si un préstamo será pagado o no.
clf1.fit(X_train_scaled, y_train)

LogisticRegression(random_state=0)

In [51]:
# Evaluación
# Impute and scale X_test using the fitted imputer and scaler
X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

print("Precision del clasificador en fase de entrenamiento", clf1.score(X_train_scaled, y_train))
y_pred = clf1.predict(X_test_scaled)

Precision del clasificador en fase de entrenamiento 0.8557434695244475


In [53]:
# Reporte y Matriz
print("\nReporte de métricas del clasificador: \n", classification_report(y_test, y_pred, target_names=["No Pagado", "Pagado"]))
print("Precisión:", clf1.score(X_test_scaled, y_test))

matriz_sin_balanceo = confusion_matrix(y_test, y_pred)
print(f'Matriz Confusion:\n', matriz_sin_balanceo)


Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.14      0.00      0.00      1220
      Pagado       0.85      1.00      0.92      6744

    accuracy                           0.85      7964
   macro avg       0.49      0.50      0.46      7964
weighted avg       0.74      0.85      0.78      7964

Precisión: 0.8461828227021597
Matriz Confusion:
 [[   1 1219]
 [   6 6738]]


In [55]:
# Creación y entrenamiento con peso de clases balanceado
clf2 = LogisticRegression(random_state=0, class_weight="balanced")
clf2.fit(X_train_scaled, y_train)

# Evaluación
print("Exactitud del modelo de entrenamiento:", clf2.score(X_train_scaled, y_train))
y_pred2 = clf2.predict(X_test_scaled)

# Reporte y Matriz
print("\nReporte de métricas del clasificador: \n", classification_report(y_test, y_pred2, target_names=["No Pagado", "Pagado"]))
print("Precisión:", clf2.score(X_test_scaled, y_test))

matriz_balanceo = confusion_matrix(y_test, y_pred2)
print(f'Matriz Confusion:\n', matriz_balanceo)

Exactitud del modelo de entrenamiento: 0.6362190221031481

Reporte de métricas del clasificador: 
               precision    recall  f1-score   support

   No Pagado       0.22      0.59      0.32      1220
      Pagado       0.89      0.63      0.74      6744

    accuracy                           0.62      7964
   macro avg       0.56      0.61      0.53      7964
weighted avg       0.79      0.62      0.68      7964

Precisión: 0.6230537418382722
Matriz Confusion:
 [[ 722  498]
 [2504 4240]]


In [62]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clf2, X_test_scaled, y_test, n_repeats=30, random_state=42)
importancia = pd.DataFrame({
"Variable": X.columns,
"Importancia Media": result.importances_mean,
"Desviación": result.importances_std
}).sort_values("Importancia Media", ascending=False)
print(importancia)

                Variable  Importancia Media  Desviación
2               int_rate           0.027126    0.003201
1         loan_term_year           0.013142    0.002376
6    home_ownership_code           0.000607    0.000887
7             annual_inc           0.000473    0.003085
0            funded_amnt          -0.000326    0.000452
8                    dti          -0.000373    0.000899
5        addr_state_code          -0.001046    0.000810
10  pub_rec_bankruptcies          -0.001461    0.000865
4           purpose_code          -0.002775    0.001971
3             grade_code          -0.003411    0.000724
9             revol_util          -0.007697    0.001734


In [64]:
# Visualizar importancias con Plotly
import plotly.express as px
fig4 = px.bar(importancia, x="Variable", y="Importancia Media",
error_y="Desviación", title="Importancia de características (Regresión Logística - Permutation)",
text_auto=".3f", color="Variable")
fig4.update_layout(width=800, height=600)
fig4.show()

In [66]:
# Re-división con stratify y normalización
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Imputar valores antes de escalar
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Modelo Escalado
RL_scaled = LogisticRegression(random_state=0, class_weight='balanced')
RL_scaled.fit(X_train_scaled, y_train)

# Ajuste de Umbral (Ejemplo con 0.45)
umbral = 0.45
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
y_pred_umbral = (y_prob >= umbral).astype(int)

# Evaluación del modelo con umbral ajustado
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.45
              precision    recall  f1-score   support

   No Pagado       0.25      0.49      0.33      1177
      Pagado       0.89      0.74      0.81      6787

    accuracy                           0.70      7964
   macro avg       0.57      0.61      0.57      7964
weighted avg       0.80      0.70      0.74      7964


Matriz de Confusión:
[[ 572  605]
 [1761 5026]]


In [67]:
# Ajustar el umbral de decisión a 0.40
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.40
umbral = 0.40   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.40:")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.4
              precision    recall  f1-score   support

   No Pagado       0.28      0.38      0.32      1177
      Pagado       0.89      0.83      0.86      6787

    accuracy                           0.76      7964
   macro avg       0.58      0.61      0.59      7964
weighted avg       0.80      0.76      0.78      7964


Matriz de Confusión con umbral de decisión a 0.40:
[[ 451  726]
 [1160 5627]]


In [68]:
# Ajustar el umbral de decisión a 0.55
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.55
umbral = 0.55   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.55")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.55
              precision    recall  f1-score   support

   No Pagado       0.20      0.71      0.32      1177
      Pagado       0.91      0.52      0.66      6787

    accuracy                           0.55      7964
   macro avg       0.56      0.62      0.49      7964
weighted avg       0.81      0.55      0.61      7964


Matriz de Confusión con umbral de decisión a 0.55
[[ 838  339]
 [3262 3525]]


In [70]:
# Ajustar el umbral de decisión a 0.60
# Predicción con umbral ajustado
# Extraer probabilidades de clase positiva (Pagado = 1)
y_prob = RL_scaled.predict_proba(X_test_scaled)[:, 1]
# Ajustar el umbral de decisión a 0.60
umbral = 0.60   # ↓ bajarlo aumenta detección de "No Pagado"
y_pred_umbral = (y_prob >= umbral).astype(int)
# Evaluación del modelo
print(f"\nReporte de clasificación con umbral = {umbral}")
print(classification_report(y_test, y_pred_umbral, target_names=["No Pagado", "Pagado"]))
print("\nMatriz de Confusión con umbral de decisión a 0.60")
print(confusion_matrix(y_test, y_pred_umbral))


Reporte de clasificación con umbral = 0.6
              precision    recall  f1-score   support

   No Pagado       0.19      0.83      0.31      1177
      Pagado       0.93      0.40      0.56      6787

    accuracy                           0.46      7964
   macro avg       0.56      0.61      0.43      7964
weighted avg       0.82      0.46      0.52      7964


Matriz de Confusión con umbral de decisión a 0.60
[[ 972  205]
 [4099 2688]]
